# ============================================================
# ONLINE RETAIL II - DATA CLEANING & PREPROCESSING
# ============================================================
# Dataset: UK-based Online Retail (Non-Store)
# Period: January 1, 2010 - November 30, 2011
# Original Records: 1,067,371
# Clean Records: 729,751
# ============================================================

In [ ]:
!pip install kaggle

In [ ]:
!kaggle datasets list

In [ ]:
!kaggle datasets download -d lakshmi25npathi/online-retail-dataset

In [ ]:
import os
import zipfile

zip_file_path = "online-retail-dataset.zip"
extraction_target = "data"

if os.path.exists(zip_file_path):
    try:
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(extraction_target)
        print(f"Successfully extracted {zip_file_path} to the '{extraction_target}' folder.")
    except zipfile.BadZipFile:
        print(f"Error: The file '{zip_file_path}' is corrupted. Please re-download.")

else:
    print(f"Error: '{zip_file_path}' not found. Check your Kaggle download step.")
    

##Load into pandas

In [ ]:
import pandas as pd

df = pd.read_excel("data/online_retail_II.xlsx")
df.head()

In [ ]:
xls = pd.ExcelFile("data/online_retail_II.xlsx")
print(xls.sheet_names)

Use Latest Year Sheet

In [ ]:
df1 = pd.read_excel(
    "data/online_retail_II.xlsx",
    sheet_name='Year 2009-2010'
)
df2 = pd.read_excel(
    "data/online_retail_II.xlsx",
    sheet_name='Year 2010-2011'
)

df = pd.concat([df1,df2], ignore_index= True)

df.head()

## Data Understanding

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

## Standardize Column Names

In [ ]:
df.columns = df.columns.str.strip().str.replace(' ','_')
df.columns

In [ ]:
df.columns

In [ ]:
df = df[
    (df['InvoiceDate'] >= '2010-01-01') &
    (df['InvoiceDate'] < '2011-12-01')
]

In [ ]:
print(df.shape)
print(df['InvoiceDate'].min(), df['InvoiceDate'].max())

In [ ]:
df.isnull().sum()

## Remove Missing Customer ID & Description Null

In [ ]:
df = df.dropna(subset=['Customer_ID', 'Description'])

df['Customer_ID'] = df['Customer_ID'].astype('Int64')

In [ ]:
## Validate Cancellations 
canceled_invoices = df['Invoice'].astype(str).str.startswith('C')
all_canceled_have_negative_qty = df[canceled_invoices]['Quantity'].max() <= 0
outliers = df[(df['Quantity'] < 0) & (~canceled_invoices)]

print(f"Are all cancellations caught by Quantity filter?: {all_canceled_have_negative_qty}")
print(f"Negative quantities missing 'C' prefix: {len(outliers)}")

## Remove Invalid Transactions

In [ ]:
# Remove negative or zero quantity & zero or negative price
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

print(f"Shape after removing cancelled & invalid transactions: {df.shape}")
print(f"Total rows after cleaning: {df.shape[0]}")

In [ ]:
df['Customer_ID'] = df['Customer_ID'].astype(int)

## Create Revenue Column

In [ ]:
df['TotalPrice'] = df['Quantity']*df['Price']

print("Quantity statistics:")
print(df['Quantity'].describe())
print("\nPrice statistics:")
print(df['Price'].describe())
print("\nTotalPrice statistics:")
print(df['TotalPrice'].describe())

## Final Validation

In [ ]:
df.sort_values(by='TotalPrice', ascending=False).head(10)

In [ ]:
df[df['StockCode'].isin(['M','POST'])]['StockCode'].value_counts()

In [ ]:
# Removing the suspicious entries
df = df[~df['StockCode'].isin(['M','POST'])]

print(f"Shape after removing the data error: {df.shape}")
print(f"Rows removed: {757490 - df.shape[0]}")

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.shape

In [ ]:
# Check final dataset
print(df.info())
print("\n" + "="*50)
print("Data Quality Summary:")
print(f"Total Records: {df.shape[0]}")
print(f"Date Range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"Unique Customers: {df['Customer_ID'].nunique()}")
print(f"Unique Products: {df['StockCode'].nunique()}")
print(f"Countries: {df['Country'].nunique()}")
print(f"\nMissing Values:\n{df.isnull().sum()}")

## PUSH DATA TO SQL SERVER

In [ ]:
import pyodbc
print(pyodbc.drivers())

## Create Connection

In [ ]:
from sqlalchemy import create_engine
server = r"localhost\SQLEXPRESS"


engine = create_engine(
    f"mssql+pyodbc://@{server}/OnlineRetailDB?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server",
    fast_executemany=True
)

In [ ]:
## Optimize Data Types

from sqlalchemy import  types

dtype_mapping = {
    'Invoice': types.String(20),
    'StockCode': types.String(20),
    'Description': types.String(255),
    'Quantity': types.Integer(),
    'InvoiceDate': types.DateTime(),
    'Price': types.Numeric(10,2),
    'Customer_ID': types.Integer(),
    'Country': types.String(50),
    'TotalPrice': types.Numeric(12,2)
}
                               


In [ ]:
## Push Data

df.to_sql(
    name="OnlineRetail",
    con=engine,
    if_exists='replace',
    index=False,
    dtype=dtype_mapping,
    chunksize=10000   # Important for performance
)
    

## LOAD RAW DATA (FOR CANCELLATION ANALYSIS)

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ===== LOAD RAW DATA FROM EXCEL =====
print("Loading ORIGINAL raw data from Excel...")

df1 = pd.read_excel(
    "data/online_retail_II.xlsx",
    sheet_name='Year 2009-2010'
)
df2 = pd.read_excel(
    "data/online_retail_II.xlsx",
    sheet_name='Year 2010-2011'
)

# Combine both dataframes
df = pd.concat([df1, df2], ignore_index=True)
df.rename(columns={'Customer ID': 'Customer_ID'}, inplace=True)
if 'Total_Price' not in df.columns:
    df['TotalPrice'] = df['Quantity'] * df['Price']

print("Raw data loaded successfully!")
print(f"Total Rows: {len(df)}")
print(f"Cancelled orders: {(df['Quantity'] < 0).sum()}")
print(f"Missing CustomerID: {df['Customer_ID'].isna().sum()}")
print(f"Missing Description: {df['Description'].isna().sum()}")

dtype_mapping = {
    'Invoice': types.String(20),
    'StockCode': types.String(20),
    'Description': types.String(255),
    'Quantity': types.Integer(),
    'InvoiceDate': types.DateTime(),
    'Price': types.Numeric(10,2),
    'Customer_ID': types.Integer(),
    'Country': types.String(50),
    'TotalPrice': types.Numeric(12,2)
}

# ===== PUSH TO SQL SERVER =====
engine = create_engine(f"mssql+pyodbc://@{server}/OnlineRetailDB?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server",
    fast_executemany=True)

df.to_sql('OnlineRetailRaw', con=engine, if_exists='replace', index=False, dtype=dtype_mapping, chunksize=10000)

print("Data successfully pushed to SQL Server!")